In [ ]:
import pandas as pd
import numpy as np
import pyreadr
import directories as dir
import matplotlib.pyplot as plt

# Create directory for data
data_fs = dir.ROOT / "data" / "raw_data" / "fs.csv"
data_suvr = dir.ROOT / "data" / "raw_data" / "suvr.csv"

# Parse csv
fs = pd.read_csv(data_fs)
suvr = pd.read_csv(data_suvr)

"""
----- Freesurfer data exploration -----
Key:
- CV = cortical volume 
- SA = surface area 
- TA = cortical thickness avg 
- TS = thickness stdev 
- SV = subcortical volume
"""

print("----- Freesurfer data exploration -----")
print("Shape: ", fs.shape)
print("Unique subjects (RID): ", fs['RID'].nunique())
print("QC total counts: \n", fs['OVERALLQC'].value_counts())

st_cols = [c for c in fs.columns if c.startswith("ST")] 
cv_cols = [c for c in st_cols if c.endswith("CV")] 
sa_cols = [c for c in st_cols if c.endswith("SA")] 
ta_cols = [c for c in st_cols if c.endswith("TA")] 

print("ST count: ", len(st_cols))
print("CV count: ", len(cv_cols))
print("SA count: ", len(sa_cols))
print("TA count: ", len(ta_cols))

----- Freesurfer data exploration -----
Shape:  (12167, 347)
Unique subjects (RID):  3175
QC total counts: 
 OVERALLQC
Partial             586
Pass                478
Hippocampus Only     11
Fail                  6
Name: count, dtype: int64
ST count:  326
CV count:  69
SA count:  69
TA count:  68


/var/folders/rt/m33zscwn3p9700c8mctz1h0w0000gn/T/ipykernel_29319/2397179679.py:12: DtypeWarning: Columns (0: OVERALLQC, 1: TEMPQC, 2: FRONTQC, 3: PARQC, 4: INSULAQC, 5: OCCQC, 6: BGQC, 7: CWMQC, 8: VENTQC, 9: HIPPOQC) have mixed types. Specify dtype option on import or set low_memory=False.
  fs = pd.read_csv(data_fs)


In [32]:
"""
----- PET SUVR data exploration -----
QC Results Key:
- 2  = pass
- 1  = partial pass
- 0  = fail
- -1 = not assessed
- -2 = cannot be processed 
"""

print("\n----- PET SUVR data exploration -----")
print("Shape: ", suvr.shape)
print("Unique subjects (RID): ", suvr['RID'].nunique())
print("QC total counts: \n", suvr['qc_flag'].value_counts())
print("QC = 2 counts: ", suvr[suvr['qc_flag'] == 2]['RID'].nunique())
print("QC >= 1 counts: ", suvr[suvr['qc_flag'] >= 1]['RID'].nunique())
print("QC >= 0 counts: ", suvr[suvr['qc_flag'] >= 0]['RID'].nunique())


----- PET SUVR data exploration -----
Shape:  (2489, 339)
Unique subjects (RID):  1448
QC total counts: 
 qc_flag
 2    2311
-1     137
 1      30
-2       9
 0       2
Name: count, dtype: int64
QC = 2 counts:  1375
QC >= 1 counts:  1386
QC >= 0 counts:  1388


In [36]:
"""
----- ADNI Merge data exploration ----- 
"""

result = pyreadr.read_r(dir.ROOT / "data" / "raw_data" / "ADNIMERGE2" / "data" / "DXSUM.rda")
adsl = result[list(result.keys())[0]]
print(adsl.head())

  ORIGPROT COLPROT        PTID  RID VISCODE VISCODE2    EXAMDATE DIAGNOSIS  \
0    ADNI1   ADNI1  011_S_0002  2.0      bl       bl  2005-09-29        CN   
1    ADNI1   ADNI1  011_S_0003  3.0      bl       bl  2005-09-30  Dementia   
2    ADNI1   ADNI1  011_S_0005  5.0      bl       bl  2005-09-30        CN   
3    ADNI1   ADNI1  011_S_0008  8.0      bl       bl  2005-09-30        CN   
4    ADNI1   ADNI1  022_S_0007  7.0      bl       bl  2005-10-06  Dementia   

  DXNORM DXNODEP  ... DXODES              DXCONFID    ID SITEID    USERDATE  \
0    Yes     NaN  ...    NaN      Highly Confident   2.0  107.0  2005-10-01   
1    NaN     NaN  ...    NaN  Moderately Confident   4.0  107.0  2005-10-01   
2    Yes     NaN  ...    NaN      Highly Confident   6.0  107.0  2005-10-01   
3    Yes     NaN  ...    NaN  Moderately Confident   8.0  107.0  2005-10-01   
4    NaN     NaN  ...    NaN      Highly Confident  10.0   10.0  2005-10-06   

  USERDATE2 DD_CRF_VERSION_LABEL LANGUAGE_CODE HAS_QC_ER

In [67]:
"""
----- Total Usable Data ----- 
Freesurfer Filter:
- OVERALL QC: Only exclude "Fail"
- FIELD_STRENGTH: 3T 
- NaN: No more than 20% NAN 
"""

# QC Filter
print("FS shape before filter: ", fs.shape[0])
fs_QC = fs[
    fs["OVERALLQC"].isin(["Pass", "Partial", "Hippocampus Only"]) | 
    fs["OVERALLQC"].isna()
    ]
print("FS shape after filtering for QC: ", fs_QC.shape[0])

# FS Filter
fs_QC_FS = fs_QC[fs_QC["FIELD_STRENGTH"] == "3T"]
print("FS shape after filtering for FS: ", fs_QC_FS.shape[0])

# NaN Filter (just for fun, but will implement once FS and SUVR is merged)
st_cols = [c for c in fs_QC_FS.columns if c.startswith("ST")]
cv_cols = [c for c in st_cols if c.endswith("CV")]
sa_cols = [c for c in st_cols if c.endswith("SA")]
ta_cols = [c for c in st_cols if c.endswith('TA')]
cols = cv_cols + sa_cols + ta_cols
print(fs_QC_FS[cols].isna().mean(axis=1).max())









FS shape before filter:  12167
FS shape after filtering for QC:  12161
FS shape after filtering for FS:  7681
0.9951456310679612
